# Import

In [ ]:
import numpy as np
import scipy
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import matplotlib.lines as mlines

plt.rcParams.update({'font.size': 16})

In [ ]:
from iaa_api import InterAnnotatorAgreementAPI
from competitors.data_handling_competitors import seed_everything
from utils import *
from syntethic_exps import to_LA

In [ ]:
seed_everything(seed=42)

# Transformation of the values of H in A or B

-  $A = \frac{\log{\frac{1-\nu}{\nu}} + H \log{\frac{T_{11}}{1-T_{00}}}}{\log{\frac{T_{11}T_{00}}{(1-T_{11})(1-T_{00})}}}$


- $ B = \frac{\log{\frac{\nu}{1-\nu}} + H \log{\frac{T_{00}}{1-T_{11}}}}{\log{\frac{T_{11}T_{00}}{(1-T_{11})(1-T_{00})}}}$

In [ ]:
def H_to_AB(vu, T, H:float, A: bool = True, slack: float = 1e-4):
    """
    We obtain the values of A and B from T, H and vu as defined in the report
    vu: distribution of the classes
    T: T matrix
    H: number of annotators (we assume it odd)
    A: if True it returns the A matrix, else the B matrix
    slack: is a slack variable used to avoid to compute log(0) which goes to infinity
    """
    nu_ratio = vu/(1-vu)
    delta_0 = 1-T[0][0]
    delta_1 = 1-T[1][1]
    if T[0][0] == T[1][1] and A:
        return ((-np.log(nu_ratio+slack)) / (2*np.log((T[0][0]/(delta_0))+slack))) + (H/2)
    if T[0][0] == T[1][1] and not A:
        return ((np.log(nu_ratio+slack)) / (2*np.log((T[0][0]/(delta_0))+slack))) + (H/2)
    if T[0][0] != T[1][1] and A:
        return (np.log((1/nu_ratio)+slack)+ H*np.log((T[1][1]/delta_0)+slack)) / (np.log(((T[0][0]*T[1][1])/(delta_0*delta_1))+slack))
    if T[0][0] != T[1][1] and not A:
        return (np.log(nu_ratio+slack)+ H*np.log((T[0][0]/delta_1)+slack)) / (np.log(((T[0][0]*T[1][1])/(delta_0*delta_1))+slack))

# Majority voting

$T_{cc}^{MV} = \sum_{i=\lceil{\frac{H}{2}}\rceil}^{H} \binom{H}{i} T_{cc}^{i}\,(1-T_{cc})^{H-i}$

In [ ]:
def T_majority(T, H:int):
    """
    Simple computation of MV as in the formula
    """
    T_mv = np.zeros_like(T)
    for k in range(int(((H+1)/2)),H+1):
        T_mv = T_mv + scipy.special.binom(H,k) * np.power(T,k)* \
        np.power(1-T,H-k)
    return T_mv

# Oracle Maximum A Posteriori (MAP)

$T_{cc}^{MAP} = \sum_{i=\lceil{A_c}\rceil}^{H} \binom{H}{i} T_{cc}^{i}\,(1-T_{cc})^{H-i}$

In [ ]:
def T_map(T, H:int, vu:float, debug:bool=False):
  """
    Simple computation of oracle MAP as in the formula
    """
  T_map = np.zeros_like(T)
  A = H_to_AB(vu, T, H, A=True)
  B = H_to_AB(vu, T, H, A=False)
  if debug:
    print("A: ", int(np.ceil(A)))
    print("B: ", int(np.ceil(B)))
    print("(H+1)/2: ", (H+1)/2)

  for k in range(int(np.ceil(A)),H+1):
      T_map[0][0] += scipy.special.binom(H,k) * \
      np.power(T[0][0],k)*  np.power(1-T[0][0],H-k)

    

  for k in range(int(np.ceil(B)),H+1):
      T_map[1][1] += scipy.special.binom(H,k) * \
      np.power(T[1][1],k)*  np.power(1-T[1][1],H-k)
 
  T_map[0][1] = 1-T_map[0][0]
  T_map[1][0] = 1-T_map[1][1]
  return T_map

# Method to chech when MAP is better than majority and when they are equal

If $T_{cc}^{MV} \left( \nu \,\,\,\,\,\, 1- \nu \right) < T_{cc}^{MAP} \left( \nu \,\,\,\,\,\, 1- \nu \right) \Rightarrow$ MAP is better than MV

If $T_{cc}^{MV} \left( \nu \,\,\,\,\,\, 1- \nu \right) > T_{cc}^{MAP} \left( \nu \,\,\,\,\,\, 1- \nu \right) \Rightarrow$ MV is better than MAP

If $\lVert T_{cc}^{MV} \left( \nu \,\,\,\,\,\, 1- \nu \right) - T_{cc}^{MAP} \left( \nu \,\,\,\,\,\, 1- \nu \right) \rVert < 10^{-2} \Rightarrow$ MAP = MV

In [ ]:
def MAP_better_majority(T_majority, T_map, vu:float,
                        debug:bool=False, required_difference=False):
  
  if debug:
    print( "Difference Majority-MAP: ", -np.dot(np.diag(T_majority), np.array([vu, 1-vu])) + np.dot(np.diag(T_map), np.array([vu, 1-vu])))
    print("Majority: ", np.dot(np.diag(T_majority), np.array([vu, 1-vu])))
    print("MAP: ", np.dot(np.diag(T_map), np.array([vu, 1-vu])))
    print( "Check 1. T_00 MAP > T_00 MV: ",  T_map[0][0] - T_majority[0][0] > (1-vu)/(vu)*(-T_map[1][1] + T_majority[1][1])  )
    print( "Check 2. T_11 MAP > T_11 MV ",  -T_map[0][0] + T_majority[0][0] < (1-vu)/(vu)*(T_map[1][1] - T_majority[1][1])  )
    print( " 1-nu/nu", (1-vu)/vu)
  if not required_difference:
    if np.dot(np.diag(T_majority), np.array([vu, 1-vu])) < np.dot(np.diag(T_map), np.array([vu, 1-vu])):
      return "MAP > MV"
    elif np.linalg.norm(np.dot(np.diag(T_majority), np.array([vu, 1-vu])) - np.dot(np.diag(T_map), np.array([vu, 1-vu]))) < 1e-2:
      return "MAP = MV"
    else:
      return "MAP < MV"
  else:
    return -np.dot(np.diag(T_majority), np.array([vu, 1-vu])) + np.dot(np.diag(T_map), np.array([vu, 1-vu]))

# Plots

In [ ]:
def compute_differences_mv_map(T, vu:float, max_H:int=30):
    """On the x axis there is the number of annotators while on the y axis the difference
    between MAP and MV. It is possible to notice how with H high, they are practically the same.
    T: T matrix
    H: number of annotators (we assume it odd)
    max_H: maximum number of annotators
    debug: print additional info
    focus: if True, the plot is focused on the first values of H
    """
    all_H_values = [x for x in range(1, max_H +1) if x % 2 != 0]
    differences = []

    for H in all_H_values:
        T_maj = T_majority(T,H)
        T_map_matrix = T_map(T, H, vu)
        differences.append(-np.dot(np.diag(T_maj), np.array([vu, 1-vu]))  + np.dot(np.diag(T_map_matrix), np.array([vu, 1-vu])))
    return all_H_values, differences

In [ ]:
def plot_multiple_differences(T_matrices: list, vu_values: list, savefig:bool=False):
    """Given a collection of T matrices and a collection of vu values it allows 
    to compute the difference from the method compute_differences_mv_map
    and then it generates the plots.
    """
    res = {}
    for T_single, vu_single in zip(T_matrices, vu_values):
        res[T_single[0][0]] = compute_differences_mv_map(T_single, vu_single, 100)[1]
    x_values = compute_differences_mv_map(T_matrices[0], vu_values[0], 100)[0]
    all_colors = obtain_color(len(T_matrices))
    all_markers = obtain_markers(len(T_matrices))

    fig, ax = plt.subplots(figsize=(8, 6))
    for i, (T_cc,res_val) in enumerate(res.items()):
        label =  r'$T_{{cc}} = {:.2f}, \, \nu = {:.1f},$'.format(T_cc, vu_values[i])
        plt.plot(x_values, res_val, color=all_colors[i],
                  marker=all_markers[i], label=label)
    plt.xlim(0,x_values[-1]+1)
    plt.legend()   
    plt.xlabel(r'$H$')
    plt.ylabel(r'$\text{diag}(T^{MAP})\left(\nu \,\,\, ,  1-\nu\right) - \text{diag}(T^{MV})\left(\nu \,\,\, , 1-\nu\right)$')
    plt.tight_layout()
    if savefig:
        plt.savefig('results/infinity.pdf',dpi=600, format='pdf')
    plt.show()

In [ ]:
T_matrices = [np.array([[0.53, 0.47], [0.47, 0.53]]), np.array([[0.7, 0.3], [0.3, 0.7]]),
               np.array([[0.95, 0.05], [0.05, 0.95]]),
              np.array([[0.55, 0.45], [0.45, 0.55]])]
vu_values = [0.3, 0.9, 0.5, 0.1]

plot_multiple_differences(T_matrices=T_matrices, vu_values=vu_values, savefig=True)

# Main Theorem Plots with 2-coin case

In [ ]:
def check_conditions_two_coin(H:int, vu_values = [x/10 for x in range(1,10)],
                    obtain_diff:bool=False, rel_improvement = 1e-3):
    """
    The method iterates over vu values and generates T matrices with an improvement defined by the improvement variable.
    After each T matrix is generated are computed MAP and MV and is checked when MAP is better than MV.
    A plot from this check is generated.
    It is also possible to plot the difference between MAP and MV.
    """
    assert len(vu_values) % 2== 0 or len(vu_values)%3==0, "Lenght of the array must be multiple of 2 or 3."
    index,row = 0,0
    if len(vu_values) %2 != 0:
        divisor = 3
        fig, ax = plt.subplots(int(len(vu_values)/divisor), divisor, figsize=(10,10)) 
    else:
        divisor = 2
        fig, ax = plt.subplots(int(len(vu_values)/divisor), divisor, figsize=(10,10))

    for vu in tqdm(vu_values):
        final = {}
        labels = {}
        color = None
        for t in np.arange(0.5, 1-rel_improvement, rel_improvement):
            for s in np.arange(1-rel_improvement, 0.5, -rel_improvement):
                T = np.array([[t, 1-t], [1-s, s]]) #[0.3,0.7]])
                T_matrix_map = T_map(T=T, H=H, vu=vu)
                T_matrix_mv = T_majority(T=T, H=H)
                text_to_color = MAP_better_majority(T_majority=T_matrix_mv, T_map=T_matrix_map,
                                                    vu=vu)
                
                ratio_1 = my_delta(T[1][1], T[0][0])/my_delta(T[0][0],T[1][1])
                ratio_0 = my_delta(T[0][0], T[1][1])/my_delta(T[1][1],T[0][0])
                a = (1/(ratio_1**(H/2)*np.sqrt(my_rho(T))+1)<(1-vu)) and ((1-vu) < 1/(ratio_1**(H/2)* (1/ np.sqrt(my_rho(T)))+1))
                b = (1/(ratio_0**(H/2)*np.sqrt(my_rho(T))+1)<vu) and (vu < 1/(ratio_0**(H/2)* (1/ np.sqrt(my_rho(T)))+1))
                if text_to_color == "MAP > MV": #If MV is less then MAP -> color is red
                    color = '#e41a1c'
                if text_to_color == "MAP < MV": #If MV is greater then MAP -> color is green
                    color = '#377eb8'
                if a and b: #MAP equal to MV
                    color = '#377eb8'
                final[(t,s)] = color
                labels[(t,s)] = np.dot(np.diag(T_matrix_map), np.array([vu, 1-vu])) - np.dot(np.diag(T_matrix_mv), np.array([vu, 1-vu]))
        t_00 = [key[0] for key in final.keys()]
        t_11 = [key[1] for key in final.keys()]
        if obtain_diff:
            if len(vu_values) == 2:
                ax[index].scatter(t_00, labels.values(), c=final.values())
            else:
                ax[row][index].scatter(t_00, labels.values(), c=final.values())
        else:
            if len(vu_values) == 2:
                ax[index].scatter(t_00, t_11, c=final.values())
                ax[index].set_xlabel(r'$T_{00}$')
                ax[index].set_ylabel(r'$T_{11}$')
                ax[index].set_title(r'$\nu=$' + str(vu))
            else:
                ax[row][index].scatter(t_00, t_11, c=final.values())
                ax[row][index].set_xlabel(r'$T_{00}$')
                ax[row][index].set_ylabel(r'$T_{11}$')
                ax[row][index].set_title(r'$\nu=$' + str(vu))
        if (index+1) % divisor == 0:
            index = 0
            row +=1 
        else:
            index +=1
    custom_labels = ['MAP > MV', 'MAP = MV']
    handles = [plt.Line2D([0], [0], color='#e41a1c', lw=2),
               plt.Line2D([0], [0], color='#377eb8', lw=2),]
    fig.legend(handles, custom_labels, loc='lower right', bbox_to_anchor=(0.81, 0.63), fontsize='14') 
    fig.tight_layout() 
    plt.savefig('results/plot_two_coin.pdf', format='pdf')
    plt.show()

In [ ]:
H = 3
check_conditions_two_coin(H=H, obtain_diff=False, vu_values = [x/10 for x in range(1,10)],
                          rel_improvement = 1e-3)

## Heatmap

In [ ]:
def plot_heatmap_two_coin(H:int, vu_values = [x/10 for x in range(1,10)],
                          rel_improvement = 1e-3):
    """
    The method iterates over vu values and generates T matrices with an improvement defined by the improvement variable.
    After each T matrix is generated are computed MAP and MV and is checked the difference between MAP and MV.
    A heatmap for each value of vu is generated
    """
    assert len(vu_values) % 2== 0 or len(vu_values)%3==0, "Lenght of the array must be multiple of 2 or 3."
    index,row = 0,0
    if len(vu_values) %2 != 0:
        divisor = 3
        fig, ax = plt.subplots(int(len(vu_values)/divisor), divisor, figsize=(10,10))
    else:
        divisor = 2
        fig, ax = plt.subplots(int(len(vu_values)/divisor), divisor, figsize=(10,10)) 

    for vu in tqdm(vu_values):
        final = {}
        for t in np.arange(0.5, 1-rel_improvement, rel_improvement):
            for s in np.arange(1-rel_improvement, 0.5, -rel_improvement):
                T = np.array([[t, 1-t], [1-s, s]])
                T_matrix_map = T_map(T=T, H=H, vu=vu)
                T_matrix_mv = T_majority(T=T, H=H)
                difference_map_mv = MAP_better_majority(T_majority=T_matrix_mv, T_map=T_matrix_map,
                                                    vu=vu, required_difference=True)
                final[(t,s)] = difference_map_mv
        t_00 = [key[0] for key in final.keys()]
        t_11 = [key[1] for key in final.keys()]
        if len(vu_values) == 2:
            im = ax[index].scatter(t_00, t_11, c=list(final.values()), cmap='YlOrBr')
            ax[index].set_xlabel(r'$T_{00}$')
            ax[index].set_ylabel(r'$T_{11}$')
            ax[index].set_title(r'$\nu=$' + str(vu))
        else:
            im = ax[row][index].scatter(t_00, t_11, c=list(final.values()), cmap='YlOrBr')
            ax[row][index].set_xlabel(r'$T_{00}$')
            ax[row][index].set_ylabel(r'$T_{11}$')
            ax[row][index].set_title(r'$\nu=$' + str(vu))
        if (index+1) % divisor == 0:
            index = 0
            row +=1 
        else:
            index +=1
    fig.colorbar(im, ax=ax, location='bottom', orientation = 'horizontal', anchor = (0.5, -.9))
    fig.tight_layout()
    plt.tight_layout() 
    plt.savefig('results/plot_heatmap_two_coin.pdf', format='pdf')
    plt.show()

In [ ]:
plot_heatmap_two_coin(H=3, rel_improvement=1e-2)

# Main Theorem

In [ ]:
def check_condistions_main_theorem(H:int, vu_values = [x/10 for x in range(1,10)], rel_improvement = 1e-2,
                                   estimate:str=None):
    """
    An array of vu values is iterated. For each vu value:
    Is generated a T matrix with T00 = T11 and is computed if for each T matrix is better MV or MAP.
    There are also some "region colors" which help to clarify why in a situation a method is better wrt. another one.
    H: number of annotators (we assume it odd)
    vu_values: array of distribution of samples for each class.
    len(vu_values) must be multiple of 2 or 3.
    """
    plt_name = 'standard_MV_MAP'
    final = {}
    for vu in tqdm(vu_values):
        color = None
        for t in np.arange(0.5, 1-rel_improvement, rel_improvement):
            T = np.array([[t, 1-t], [1-t, t]])
            if estimate is not None:
                true_labels = generate_true_labels(C=2, N=100, D=[vu, 1-vu])
                data = generate_annotations(true_labels, T, H=H, obtain_list=True)
                if estimate == 'iaa':
                    modified_data = np.array(data, dtype=object)
                    iaa = InterAnnotatorAgreementAPI(modified_data)
                    iaa._build_t_matrix()
                    T = iaa._t_hat
                    vu_est = iaa._label_distribution[0]
                    plt_name = 'iaa_MV_MAP'
                if estimate == 'iwmv':
                    e2wl, w2el, label_set = to_LA(data)
                    _, _, T_matrix = iwmv(e2wl, w2el, label_set, T_required=True)
                    T = T_matrix
                    plt_name = 'iwmv_MV_MAP'
            try:
                if estimate == 'iaa':
                    T_matrix_map = T_map(T=T, H=H, vu=vu_est, debug=False)
                else:
                    T_matrix_map = T_map(T=T, H=H, vu=vu, debug=False)
                T_matrix_mv = T_majority(T=T, H=H)
                text_to_color = MAP_better_majority(T_majority=T_matrix_mv, T_map=T_matrix_map,
                                                    vu=vu)
                
                if text_to_color == "MAP > MV": #If MV is less then MAP -> color is red
                    color = '#e41a1c'
                if text_to_color == "MAP < MV": #If MV is greater then MAP -> color is green
                    color = '#377eb8'
                if text_to_color == "MAP = MV": #If MV is equal to MAP -> color is yellow
                    color = '#377eb8'
                final[(t,vu)] = color
            except ValueError:
                print('+1')
                continue
    t_values = [1 - x[0] for x in final.keys()]
    vu_all = [x[1] for x in final.keys()]
    if estimate is None:
        color_map = {'#e41a1c': 'oMAP>MV', '#377eb8': 'oMAP=MV'}
    if estimate == 'iaa':
        color_map = {'#e41a1c': r'eMAP$_{IAA} > MV$', '#377eb8': r'eMAP$_{IAA} = MV$'}
    if estimate == 'iwmv':
        color_map = {'#e41a1c': r'eMAP$_{IWMV} > MV$', '#377eb8': r'eMAP$_{IWMV} = MV$'}
    fig, ax = plt.subplots(figsize=(8, 6))
    plt.scatter(t_values, vu_all, c=final.values())

    legend_artists = []
    legend_labels = []
    for color, label in color_map.items():
        dummy_line = mlines.Line2D([], [], color=color, marker='o', linestyle='None', markersize=10, label=label)
        legend_artists.append(dummy_line)
        legend_labels.append(label)
        
    plt.xlabel(r'$1-T_{cc}$')
    plt.ylabel(r'$\nu$')
    #plt.title(r'$\nu=$' + str(vu))
    plt.legend(handles=legend_artists, labels=legend_labels, loc='best')

    plt.tight_layout()
    plt.savefig(f'results/{plt_name}.pdf', dpi=300, format='pdf')
    plt.show()

In [ ]:
estimates = [None, 'iaa', 'iwmv']
for est in estimates:
    check_condistions_main_theorem(H = 3, vu_values=np.linspace(start=0.0001, stop=0.999, num=50), rel_improvement=1e-3,
                               estimate=est)

## Heatmap

In [ ]:
def plot_heatmap_difference(H:int, vu_values = [x/10 for x in range(1,10)],
                            rel_improvement = 1e-3):
    final = {}
    for vu in vu_values:
        for t in np.arange(0.5, 1-rel_improvement, rel_improvement):
            T = np.array([[t, 1-t], [1-t, t]])
            T_matrix_map = T_map(T=T, H=H, vu=vu, debug=False)
            T_matrix_mv = T_majority(T=T, H=H)
            differce_map_mv = MAP_better_majority(T_majority=T_matrix_mv, T_map=T_matrix_map,
                                                vu=vu, required_difference=True)
            final[(t,vu)] = differce_map_mv

    t_values = [1 - x[0] for x in final.keys()]
    vu_all = [x[1] for x in final.keys()]
    fig, ax = plt.subplots(figsize=(8, 6))
    im = plt.scatter(t_values, vu_all, c=list(final.values()), cmap='YlOrBr')
    cbar = fig.colorbar(im, ax=ax)

    plt.xlabel(r'$1-T_{cc}$')
    plt.ylabel(r'$\nu$')

    plt.tight_layout()
    plt.savefig('results/plot_heatmap.pdf', format='pdf')
    plt.show()

In [ ]:
vu_values = np.arange(0., 1., 0.01)
plot_heatmap_difference(3, vu_values=vu_values,rel_improvement = 1e-3)

# Condition with more than 2 classes

In [ ]:
def enumerate_simplex(C, H):
    """Enumerate all n in Delta_H = {n in N_0^C : sum(n) = H}."""
    if C == 1:
        yield (H,)
        return
    for n1 in range(H + 1):
        for rest in enumerate_simplex(C - 1, H - n1):
            yield (n1,) + rest


def compute_K(T, H):
    """
    Compute K_{ij}^{(H)} for all i != j.

    K_{ij}^{(H)} = max_{n in A_i} prod_ell (T_{j,ell}/T_{i,ell})^{n_ell}

    where A_i = {n in Delta_H : n_i >= n_ell for all ell}.

    Parameters
    ----------
    T : ndarray of shape (C, C)
        Confusion/transition matrix. T[i, ell] = T_{i,ell}.
    H : int
        Total number of votes.

    Returns
    -------
    K : ndarray of shape (C, C)
        K[i, j] = K_{ij}^{(H)}. Diagonal is 1.
    """
    C = T.shape[0]
    log_ratio = np.zeros((C, C, C))
    for i in range(C):
        for j in range(C):
            if i == j:
                continue
            for ell in range(C):
                log_ratio[i, j, ell] = np.log(T[j, ell] / T[i, ell])

    log_K = np.full((C, C), -np.inf)
    np.fill_diagonal(log_K, 0.0)

    for n in enumerate_simplex(C, H):
        n_arr = np.array(n, dtype=float)
        majority_indices = [i for i in range(C) if n[i] == max(n)]

        for i in majority_indices:
            for j in range(C):
                if i == j:
                    continue
                val = np.dot(n_arr, log_ratio[i, j])
                if val > log_K[i, j]:
                    log_K[i, j] = val

    return np.exp(log_K)


def check_condition(T, H, nu, tol=1e-12):
    """
    Check whether the given nu satisfies Theorem 1's condition:
        nu_i >= K_{ij}^{(H)} * nu_j   for all i != j.

    Parameters
    ----------
    T : ndarray of shape (C, C)
        Confusion/transition matrix.
    H : int
        Total number of votes.
    nu : ndarray of shape (C,)
        Prior probability vector.
    tol : float
        Tolerance for the comparison.

    Returns
    -------
    ok : bool
        True if the condition holds for all pairs.
    details : list of tuples
        Each entry is (i, j, lhs, rhs, pair_ok) where
        lhs = nu[i], rhs = K_{ij}^{(H)} * nu[j],
        pair_ok = (lhs > rhs + tol).
    """
    C = T.shape[0]
    K = compute_K(T, H)

    ok = True
    details = []
    for i in range(C):
        for j in range(C):
            if i == j:
                continue
            lhs = nu[i]
            rhs = K[i, j] * nu[j]
            pair_ok = lhs > rhs + tol
            details.append((i, j, lhs, rhs, pair_ok))
            ok = ok and pair_ok

    return ok, details